# Experiment 1.3.8 — SNN Local Feature Preservation

## Scientific question

Experiment 1.3.7 showed that the current SNN representation contains **complementary information beyond coarse Raw250 counts**, but SNN-only performance is less stable and does not reliably preserve all of the robust count-based information.

The goal here is **not** to keep a Raw+SNN fusion branch at inference. The final target remains an SNN-only local feature extractor:

\[
X_{\mathrm{spike}} \rightarrow \mathrm{continuous\ SNN} \rightarrow z_1,z_2,\ldots,z_B,
\qquad z_b\in\mathbb{R}^{64}.
\]

For every 250 ms window, the L3 local feature is

\[
z_b=\sum_{t\in W_b}S_t^{L3}.
\]

The SNN state remains continuous across the full gesture and is **not reset at 250 ms boundaries**. The bins are readout windows only.

### Hypothesis

The current SNN can learn useful temporal information, but it may discard part of the robust local channel-semantic structure present in Raw250 counts. Explicit auxiliary objectives should make that information easily decodable from the 64-D latent feature without forcing individual SNN neurons to correspond to individual raw channels.

## Representation-shaping objectives

### 1. Whole-gesture classification

As in Experiment 1.3.5, all 250 ms local features are concatenated and a whole-gesture classifier supplies

\[
L_{\mathrm{cls}}=CE(\hat y,y).
\]

### 2. Semantic count preservation

For each 250 ms input window:

\[
c_b[c]=\sum_{t\in W_b}X_{t,c}\in\mathbb{R}^{30}.
\]

Raw channels have fixed semantics, whereas the 64 SNN neurons form a distributed latent representation. Therefore we **do not** impose `z[:30] ≈ c`. Instead a training-only linear decoder learns

\[
\hat c_b=W_cz_b+b_c,
\]

which requires the robust 30-channel information to remain **linearly decodable** from the 64-D SNN feature. Counts are transformed with `log1p`, standardized per channel using train users only, and trained with masked SmoothL1 loss. Partial valid bins are weighted by their valid fraction and padded bins are ignored.

### 3. Within-window temporal preservation

Each 250 ms window is divided into early and late 125 ms halves. Let their channel counts be `q_E` and `q_L`. For an active channel:

\[
r_b[c]=\frac{q_b^E[c]-q_b^L[c]}{q_b^E[c]+q_b^L[c]+\epsilon}\in[-1,1].
\]

A second training-only linear decoder predicts this normalized temporal contrast. Temporal loss is computed only for fully valid 250 ms bins and channels with non-zero activity.

### Full objective

\[
\boxed{L=L_{\mathrm{cls}}+\lambda_cL_{\mathrm{count}}+\lambda_tL_{\mathrm{temp}}}
\]

The semantic decoder, temporal decoder, and global classifier are training/evaluation tools. The final deployed output is only `z_b`.

In [3]:
from pathlib import Path

def find_repo_root(start=None):
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
SCRIPT_DIR = REPO_ROOT / 'scripts/experiment_1_3_8_local_feature_preservation'
print('Repository root:', REPO_ROOT)
print('Implementation:', SCRIPT_DIR)


Repository root: /home/ted/project/writingRing
Implementation: /home/ted/project/writingRing/scripts/experiment_1_3_8_local_feature_preservation


## 1. Data, split, and auxiliary targets

This cell loads the same 64 Hz / 30-channel unsigned event dataset and fixed `SPLIT_SEED=12345` user-disjoint split used by Experiments 1.3.5–1.3.7. It also constructs the semantic-count and early/late temporal targets using train-only normalization statistics.

In [4]:
path = SCRIPT_DIR / '01_setup.py'
exec(compile(path.read_text(), str(path), 'exec'), globals())


Repository root: /home/ted/project/writingRing
Device: cuda
Experiment: experiment_1_3_8_snn_local_feature_preservation
samples=853, users=20, classes=12
labels: ['A', 'B', 'C', 'D', 'E', 'G', 'H', 'I', 'J', 'K', 'L', 'X']
train/val/test: (632, 256, 30) (126, 256, 30) (95, 256, 30)
readout: 16 samples = 250.0 ms
{'train_users': ('user_0', 'user_1', 'user_11', 'user_12', 'user_14', 'user_19', 'user_2', 'user_3', 'user_4', 'user_5', 'user_6', 'user_7', 'user_8', 'user_9'), 'val_users': ('user_15', 'user_16', 'user_20'), 'test_users': ('user_10', 'user_13', 'user_18')}
semantic target mean range: 0.0356464609503746 0.24721387028694153
semantic target std range: 0.07761354744434357 0.2478490024805069
train temporal active elements: 50332


## 2. Model, masked objectives, training, and fresh SNN-only probe

The backbone is intentionally unchanged from Experiment 1.3.5:

\[30\rightarrow128\rightarrow128\rightarrow64\]

with shifts `((2,3), (2,3), (2,))`, `tau_mem=22.54 ms`, threshold `0.5`, and continuous state. The two auxiliary decoders are both `Linear(64, 30)` and exist only to shape the latent representation during training.

In [5]:
path = SCRIPT_DIR / '02_model_training.py'
exec(compile(path.read_text(), str(path), 'exec'), globals())


Raw250 validation-selected C: 0.1
Raw250 val BA: 0.713260582010582
Test remains untouched during development.


## 3. Development sweep and final runs

### Conditions

- **A — `cls_only`**: `L_cls`
- **B — `cls_count`**: `L_cls + lambda_c L_count`
- **C — `cls_count_temp`**: `L_cls + lambda_c L_count + lambda_t L_temp`

### Hyperparameter protocol

Development uses seed 11 only. First sweep `lambda_c ∈ {0.05, 0.10, 0.30}`, select by **validation SNN-only Logistic Regression BA**; then lock it and sweep `lambda_t ∈ {0.05, 0.10, 0.30}` using the same criterion. Auxiliary reconstruction loss is not the selection metric.

The test set is not evaluated during development. After both lambdas are locked, final training runs A/B/C for seeds `(11, 23, 101)`.

### Primary metric

After training, freeze the SNN and ignore all training heads. Flatten the `16 × 64` SNN local-feature map and train a fresh `StandardScaler + LogisticRegression`; choose `C ∈ {1e-3,1e-2,1e-1,1,10}` on validation only, then report test balanced accuracy. This SNN-only probe is the primary measure of local-feature quality.

In [6]:
path = SCRIPT_DIR / '03_run_primary.py'
exec(compile(path.read_text(), str(path), 'exec'), globals())


/home/ted/project/writingRing/scripts/experiment_1_3_8_local_feature_preservation/02_model_training.py:120: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  tensors = [torch.from_numpy(a) for a in SPLIT_ARRAYS[split]]


cls_count        seed= 11 epoch=  1 | train BA=0.0748 val BA=0.0903 | L=2.5050 (cls=2.4856, count=0.3882, temp=0.4762) | L3 FR=0.0001
cls_count        seed= 11 epoch= 10 | train BA=0.3172 val BA=0.2214 | L=1.9182 (cls=1.8988, count=0.3878, temp=0.6573) | L3 FR=0.0096
cls_count        seed= 11 epoch= 20 | train BA=0.6977 val BA=0.4243 | L=0.9099 (cls=0.8909, count=0.3795, temp=1.0968) | L3 FR=0.0316
cls_count        seed= 11 epoch= 30 | train BA=0.8561 val BA=0.4870 | L=0.5008 (cls=0.4832, count=0.3525, temp=1.5566) | L3 FR=0.0534
cls_count        seed= 11 epoch= 40 | train BA=0.8802 val BA=0.5023 | L=0.3734 (cls=0.3564, count=0.3393, temp=1.6712) | L3 FR=0.0601
cls_count        seed= 11 epoch= 50 | train BA=0.9453 val BA=0.4966 | L=0.2049 (cls=0.1885, count=0.3280, temp=1.7542) | L3 FR=0.0652
cls_count        seed= 11 epoch= 60 | train BA=0.9624 val BA=0.4920 | L=0.1585 (cls=0.1423, count=0.3242, temp=1.8678) | L3 FR=0.0748
cls_count        seed= 11 epoch= 70 | train BA=0.9635 val BA=0

,stage,lambda_count,lambda_temp,best_epoch,global_val_BA,probe_selected_C,probe_val_BA
0,count,0.05,0.0,84,0.614881,0.10,0.600992
1,count,0.10,0.0,93,0.650860,0.01,0.638161
2,count,0.30,0.0,79,0.633036,0.01,0.625860


cls_count_temp   seed= 11 epoch=  1 | train BA=0.0748 val BA=0.0833 | L=2.5483 (cls=2.4857, count=0.3882, temp=0.4757) | L3 FR=0.0001


KeyboardInterrupt: 

## 4. Secondary diagnostics

Three diagnostics help interpret the primary SNN-only result:

1. **Fresh semantic linear probe:** after training, a new linear regressor measures how well 30-channel semantic counts can be decoded from `z_b`; it does not reuse the training decoder.
2. **Fresh temporal linear probe:** measures how well early-vs-late temporal contrast remains linearly decodable.
3. **Raw250 + SNN diagnostic fusion:** retained only to test whether temporal complementarity survives. It is **not** the intended inference architecture.

Firing rates and dead-neuron fractions are also saved so any accuracy gain can be checked against changes in the firing regime.

In [ ]:
path = SCRIPT_DIR / '04_diagnostics.py'
exec(compile(path.read_text(), str(path), 'exec'), globals())


## 5. Interpretation rules

### Strong success
`cls_count_temp` gives a stable SNN-only BA above Raw250 across seeds. This supports the claim that the SNN can preserve robust channel-semantic information while adding useful temporal structure.

### Partial success
SNN-only mean BA approaches Raw250 and seed-to-seed variance decreases substantially relative to `cls_only`. The preservation objectives improve representation robustness even if they do not yet beat Raw250.

### Count-copy failure mode
Semantic decodability becomes very strong, SNN-only performance approaches Raw250, but Raw+SNN no longer improves over Raw. The SNN may have collapsed toward a count-like representation; reduce `lambda_c` or redesign/strengthen temporal preservation.

### Temporal-objective failure mode
If `cls_count_temp` is worse than `cls_count`, inspect temporal-mask coverage, the fresh temporal probe, and whether the 125 ms early/late target is too restrictive.

### Firing-regime failure mode
If gains require a large increase in L3 firing rate or a large loss of sparsity, quantify that trade-off before adopting the objective.

The final intended representation always remains:

\[
\boxed{X_{\mathrm{spike}}\rightarrow SNN\rightarrow z_1,z_2,\ldots,z_B}
\]
